# Word Embedding
- How exactly word embeddings are computed. 
- There are two techniques for this 
    - (1) supervised learning 
    - (2) self supervised learning techniques such as word2vec, glove. 
    
- Here we will look at the first technique of supervised learning by using keras embedding class

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding  # This class is to build the 1st embedding layer to craete embedding vector


In [2]:
# Here sample data of different word length is taken
reviews = ['nice food',
        'amazing restaurant',
        'too good',
        'just loved it!',
        'will go again',
        'horrible food',
        'never go there',
        'poor service',
        'poor quality',
        'needs improvement']

#This is Output or sentiment info of above statement 1 means good, 0 means bad/poor
sentiment = np.array([1,1,1,1,1,0,0,0,0,0])

In [4]:
# Here sample one hot encoding applying way. Here 50 means max 50 unique words in doc and it applies no within 50
one_hot("amazing restaurant",30)

[15, 29]

In [5]:
vocab_size = 30
encoded_reviews = [one_hot(d, vocab_size) for d in reviews]
print(encoded_reviews)

[[5, 28], [15, 29], [6, 23], [7, 16, 29], [18, 25, 17], [15, 28], [6, 25, 19], [1, 12], [1, 20], [22, 18]]


> Here we can see, few list are of length  2 and few are of length 3 or 4. So below we are adding pads and making all list length to 4
> padding='post' means padding 0 at the end

In [6]:
max_length = 4
padded_reviews = pad_sequences(encoded_reviews, maxlen=max_length, padding='post')
print(padded_reviews)

[[ 5 28  0  0]
 [15 29  0  0]
 [ 6 23  0  0]
 [ 7 16 29  0]
 [18 25 17  0]
 [15 28  0  0]
 [ 6 25 19  0]
 [ 1 12  0  0]
 [ 1 20  0  0]
 [22 18  0  0]]


> Here 'nice food' converted as [ 5 28  0  0], so nice -> 5 food -> 28

#### Here we are adding 1st layer as Embading layer
> making Embedding vector size =5 by using func Embedding from Sequential()
> # This Embedding class is to build the 1st embedding layer to craete embedding vector

![image-2.png](attachment:image-2.png)

In [8]:
embeded_vector_size = 5

model = Sequential()
model.add(Embedding(vocab_size, embeded_vector_size, input_length=max_length,name="embedding"))  # 1st Embbeding layer
model.add(Flatten()) # Then flatten Func
model.add(Dense(1, activation='sigmoid')) # Dense layer, Sigmoid Funct to predict yhat

In [19]:
X = padded_reviews 
y = sentiment

In [20]:
X 

array([[ 5, 28,  0,  0],
       [15, 29,  0,  0],
       [ 6, 23,  0,  0],
       [ 7, 16, 29,  0],
       [18, 25, 17,  0],
       [15, 28,  0,  0],
       [ 6, 25, 19,  0],
       [ 1, 12,  0,  0],
       [ 1, 20,  0,  0],
       [22, 18,  0,  0]])

In [21]:
y

array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

In [22]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 4, 5)              150       
_________________________________________________________________
flatten_1 (Flatten)          (None, 20)                0         
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 21        
Total params: 171
Trainable params: 171
Non-trainable params: 0
_________________________________________________________________
None


In [24]:
#Fit the model
model.fit(X, y, epochs=50, verbose=0)

In [25]:
# evaluate the model
loss, accuracy = model.evaluate(X, y)
accuracy

1/1 [==============================] - 0s 212ms/step - loss: 0.5832 - accuracy: 1.0000


1.0

In [30]:
# name="embedding" was named during model build for embedding layer, that name used here to get Embedding Vector value
model.get_layer('embedding').get_weights()[0]


array([[ 0.06868798,  0.12446205, -0.08290716, -0.1356599 ,  0.06744745],
       [-0.08762093,  0.08908164,  0.1328425 ,  0.09052403,  0.11735857],
       [ 0.00439036,  0.04079989,  0.02379246,  0.02416147, -0.03380138],
       [ 0.04465747, -0.03003341, -0.00294465,  0.02158216,  0.00569981],
       [ 0.02990848, -0.0101338 , -0.01965593,  0.00098941,  0.04774834],
       [ 0.06065051, -0.08443996, -0.07651015, -0.15126285, -0.09389547],
       [-0.01301487, -0.01447474,  0.03974587,  0.00516077,  0.0216159 ],
       [ 0.1299439 , -0.08262921, -0.12325875, -0.13965724, -0.12615153],
       [ 0.02960893, -0.04018705, -0.04422626,  0.02911452,  0.03673042],
       [-0.02628317, -0.02732337,  0.03212596, -0.04995004, -0.04987064],
       [ 0.02411975,  0.04826942,  0.02746418,  0.04301685, -0.04858254],
       [ 0.00579945,  0.02559488, -0.00075249, -0.03515047,  0.02525398],
       [ 0.06028906,  0.08224343, -0.11052275,  0.05414215,  0.11409707],
       [ 0.00323274, -0.01654391, -0.0

In [ ]:
weights = model.get_layer('embedding').get_weights()[0]
len(weights)

In [32]:
#Weight or Embedding Vector value of 5th encoding  which is 'nice'
#  Here 'nice food' converted as [ 5 28  0  0] in earlier encoding time so nice -> 5 food -> 28
weights[5]

array([ 0.06065051, -0.08443996, -0.07651015, -0.15126285, -0.09389547],
      dtype=float32)

In [33]:
#Weight or Embedding Vector value of 28th encoding  which is 'food'
weights[28]

array([-0.01896702,  0.04524701, -0.03654981,  0.03778649,  0.02565967],
      dtype=float32)

In [36]:
#  Here 'amazing restaurant' converted as [ 15 29  0  0] in earlier encoding time so amazing -> 15 restaurant -> 29

weights[15]

array([-0.05247675,  0.02658931,  0.02165269,  0.07110664,  0.10561047],
      dtype=float32)

In [35]:
weights[29]

array([-0.13770708, -0.08566258,  0.10704067, -0.10213562, -0.13943715],
      dtype=float32)